<a href="https://colab.research.google.com/github/MahiDaksh/mahi/blob/main/EEGContrastNet_MultiDataset_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EEG-ContrastNet — Multi-Dataset Comparison

Runs EEG-ContrastNet on **3 benchmark EEG datasets** and compares with the published Thinking Out Loud results.

| Dataset | Subjects | Classes | Chance |
|---|---|---|---|
| Thinking Out Loud (published) | 10 | 4 (inner speech) | 25% |
| PhysioNet MI | 15 | 4 (L/R/Hands/Feet) | 25% |
| BCI-IV 2a | 9 | 4 (L/R/Feet/Tongue) | 25% |
| Lee2019 OpenBMI | 15 | 2 (L/R hand) | 50% |

**Set runtime to GPU: Runtime → Change runtime type → T4 GPU**

In [1]:
# ── Cell 1: Check GPU ─────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

GPU available: True
Device: Tesla T4


In [2]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q mne pyriemann moabb scikit-learn seaborn matplotlib
print('All packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.6/153.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.7/837.7 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.1/257.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 45.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
All packages installed.


In [3]:
# ── Cell 3: Mount Google Drive (to save results) ───────────────
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/BCI_Results'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Results will be saved to:', SAVE_DIR)

Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/BCI_Results


In [4]:
# ── Cell 4: Imports & environment ─────────────────────────────
import os, sys, time, pickle, warnings
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder

from pyriemann.estimation import Covariances
try:
    from pyriemann.geometry.mean import mean_riemann
    from pyriemann.geometry.base import invsqrtm, sqrtm
except ImportError:
    from pyriemann.utils.mean import mean_riemann
    from pyriemann.utils.base import invsqrtm, sqrtm
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM

torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

RESULTS_DIR = Path(SAVE_DIR)
CKPT_DIR    = RESULTS_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

Using device: cuda


In [5]:
# ── Cell 5: Config & Dataset registry ────────────────────────
CFG = dict(
    pretrain_epochs = 100,
    pretrain_lr     = 3e-4,
    pretrain_bs     = 64,
    tau             = 0.07,
    finetune_epochs = 50,
    finetune_lr     = 5e-5,
    finetune_bs     = 32,
    ewc_lambda      = 400,
    embed_dim       = 128,
    proj_dim        = 64,
    dropout         = 0.3,
)

DATASET_REGISTRY = {
    'PhysionetMI': dict(
        moabb_class = 'PhysionetMI',
        n_subjects  = 15,
        events      = ['left_hand', 'right_hand', 'hands', 'feet'],
        n_classes   = 4,
        class_names = ['Left Hand', 'Right Hand', 'Hands', 'Feet'],
        fmin=1., fmax=40., tmin=0., tmax=2.5, resample=256,
    ),
    'BCIC4_2a': dict(
        moabb_class = 'BNCI2014_001',
        n_subjects  = 9,
        events      = None,
        n_classes   = 4,
        class_names = ['Left Hand', 'Right Hand', 'Feet', 'Tongue'],
        fmin=1., fmax=40., tmin=0.5, tmax=2.5, resample=256,
    ),
    'Lee2019_MI': dict(
        moabb_class = 'Lee2019_MI',
        n_subjects  = 15,
        events      = None,
        n_classes   = 2,
        class_names = ['Left Hand', 'Right Hand'],
        fmin=1., fmax=40., tmin=0., tmax=2.5, resample=256,
    ),
}
print('Config ready.')

Config ready.


In [6]:
# ── Cell 6: Model definition ──────────────────────────────────

class RiemannianAligner:
    def __init__(self, ref='grand_mean'):
        self.ref = ref; self.ref_ = None; self.ts_ = None
    def fit(self, covs_list):
        all_c = np.concatenate(covs_list)
        self.ref_ = mean_riemann(all_c) if self.ref == 'grand_mean' else np.eye(all_c.shape[1])
        aligned = np.concatenate([self._align(c) for c in covs_list])
        self.ts_ = TangentSpace(metric='riemann').fit(aligned)
        return self
    def _align(self, covs):
        M = mean_riemann(covs)
        W = sqrtm(self.ref_) @ invsqrtm(M)
        return np.stack([W @ C @ W.T for C in covs])
    def transform(self, covs): return self.ts_.transform(self._align(covs))
    def fit_transform(self, covs_list):
        self.fit(covs_list)
        return np.concatenate([self.ts_.transform(self._align(c)) for c in covs_list])


class EEGNet(nn.Module):
    def __init__(self, n_ch, n_t, F1=8, D=2, F2=16, embed_dim=128, drop=0.25):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(1,F1,(1,64),padding=(0,32),bias=False), nn.BatchNorm2d(F1))
        self.b2 = nn.Sequential(nn.Conv2d(F1,F1*D,(n_ch,1),groups=F1,bias=False),
                                 nn.BatchNorm2d(F1*D), nn.ELU(), nn.AvgPool2d((1,4)), nn.Dropout(drop))
        self.b3 = nn.Sequential(nn.Conv2d(F1*D,F2,(1,16),padding=(0,8),bias=False),
                                 nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1,8)), nn.Dropout(drop))
        with torch.no_grad():
            flat = self.b3(self.b2(self.b1(torch.zeros(1,1,n_ch,n_t)))).numel()
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(flat, embed_dim))
    def forward(self, x):
        if x.dim()==3: x=x.unsqueeze(1)
        return self.fc(self.b3(self.b2(self.b1(x))))


class RiemannMLP(nn.Module):
    def __init__(self, in_dim, embed_dim=128, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(drop),
            nn.Linear(256,256),    nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(drop),
            nn.Linear(256,embed_dim))
    def forward(self, x): return self.net(x)


class ProjHead(nn.Module):
    def __init__(self, d=128, p=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,d), nn.ReLU(), nn.Linear(d,p))
    def forward(self, x): return F.normalize(self.net(x), dim=-1)


class SupConLoss(nn.Module):
    def __init__(self, tau=0.07): super().__init__(); self.tau=tau
    def forward(self, feats, labels):
        B, dev = feats.size(0), feats.device
        sim = torch.mm(feats, feats.T)/self.tau
        leq = labels.unsqueeze(0)==labels.unsqueeze(1)
        eye = torch.eye(B,device=dev).bool()
        pos = leq & ~eye
        sim_max,_ = sim.max(1,keepdim=True)
        sim = sim - sim_max.detach()
        exp_sim = torch.exp(sim)*(~eye).float()
        log_prob = sim - torch.log(exp_sim.sum(1,keepdim=True)+1e-8)
        n_pos = pos.float().sum(1); valid = n_pos>0
        return (-(log_prob*pos.float()).sum(1)[valid]/n_pos[valid]).mean()


class EEGContrastNet(nn.Module):
    def __init__(self, riem_dim, n_ch, n_t, n_cls, d=128, p=64, drop=0.3):
        super().__init__()
        self.riem_enc   = RiemannMLP(riem_dim, d, drop)
        self.eeg_enc    = EEGNet(n_ch, n_t, embed_dim=d)
        self.riem_proj  = ProjHead(d, p)
        self.eeg_proj   = ProjHead(d, p)
        self.classifier = nn.Sequential(
            nn.Linear(d*2,128), nn.BatchNorm1d(128), nn.GELU(),
            nn.Dropout(drop), nn.Linear(128,n_cls))
    def forward(self, xr, xe, mode='pretrain'):
        zr=self.riem_enc(xr); ze=self.eeg_enc(xe)
        if mode=='pretrain': return self.riem_proj(zr), self.eeg_proj(ze)
        return self.classifier(torch.cat([zr,ze],dim=-1))


class BCIDataset(Dataset):
    def __init__(self, xr, xe, y):
        self.xr=torch.from_numpy(xr).float()
        self.xe=torch.from_numpy(xe).float()
        self.y =torch.from_numpy(y).long()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.xr[i], self.xe[i], self.y[i]

print('Model classes defined.')

Model classes defined.


In [7]:
# ── Cell 7: Training pipeline ─────────────────────────────────

def train_one_fold(train_subs, test_sub, n_cls, cfg, fold_idx, ds_ckpt):
    aligner  = RiemannianAligner(ref='grand_mean')
    xr_train = aligner.fit_transform([s['covs'] for s in train_subs]).astype(np.float32)
    xr_test  = aligner.transform(test_sub['covs']).astype(np.float32)
    xe_train = np.concatenate([s['X_raw'] for s in train_subs]).astype(np.float32)
    xe_test  = test_sub['X_raw'].astype(np.float32)
    y_train  = np.concatenate([s['y'] for s in train_subs])
    y_test   = test_sub['y']
    riem_dim = xr_train.shape[1]
    n_ch, n_t = test_sub['n_channels'], test_sub['n_times']

    tr_dl = DataLoader(BCIDataset(xr_train, xe_train, y_train),
                       batch_size=cfg['pretrain_bs'], shuffle=True, num_workers=2, pin_memory=True)
    te_dl = DataLoader(BCIDataset(xr_test, xe_test, y_test),
                       batch_size=64, shuffle=False, num_workers=2)

    model = EEGContrastNet(riem_dim, n_ch, n_t, n_cls,
                            d=cfg['embed_dim'], p=cfg['proj_dim'], drop=cfg['dropout']).to(DEVICE)

    # Phase 1: Contrastive pretraining
    supcon = SupConLoss(tau=cfg['tau'])
    opt1   = AdamW(model.parameters(), lr=cfg['pretrain_lr'], weight_decay=1e-4)
    sched1 = CosineAnnealingLR(opt1, T_max=cfg['pretrain_epochs'])
    model.train()
    for ep in range(cfg['pretrain_epochs']):
        for xr, xe, yb in tr_dl:
            xr,xe,yb = xr.to(DEVICE), xe.to(DEVICE), yb.to(DEVICE)
            pr,pe = model(xr, xe, mode='pretrain')
            feats  = torch.cat([pr,pe],0)
            labels = torch.cat([yb,yb],0)
            loss = supcon(feats, labels)
            opt1.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt1.step()
        sched1.step()

    # EWC Fisher
    fisher = {n: torch.zeros_like(p) for n,p in model.named_parameters() if p.requires_grad}
    model.eval()
    for xr,xe,yb in tr_dl:
        xr,xe,yb = xr.to(DEVICE), xe.to(DEVICE), yb.to(DEVICE)
        loss = F.cross_entropy(model(xr,xe,mode='classify'), yb)
        model.zero_grad(); loss.backward()
        for n,p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                fisher[n] += p.grad.data.pow(2)
    for n in fisher: fisher[n] /= max(len(tr_dl),1)
    star = {n: p.clone().detach() for n,p in model.named_parameters()}

    # Phase 2: EWC fine-tuning
    for p in model.riem_enc.parameters(): p.requires_grad=False
    val_split = int(0.8*len(y_test))
    ft_dl = DataLoader(BCIDataset(xr_test[:val_split], xe_test[:val_split], y_test[:val_split]),
                       batch_size=cfg['finetune_bs'], shuffle=True, num_workers=2)
    opt2   = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=cfg['finetune_lr'], weight_decay=1e-4)
    sched2 = CosineAnnealingLR(opt2, T_max=cfg['finetune_epochs'])
    model.train()
    for ep in range(cfg['finetune_epochs']):
        for xr,xe,yb in ft_dl:
            xr,xe,yb = xr.to(DEVICE), xe.to(DEVICE), yb.to(DEVICE)
            out = model(xr,xe,mode='classify')
            ewc_pen = sum((fisher[n]*(p-star[n]).pow(2)).sum()
                          for n,p in model.named_parameters() if p.requires_grad and n in fisher)
            loss = F.cross_entropy(out,yb) + cfg['ewc_lambda']*ewc_pen
            opt2.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt2.step()
        sched2.step()
    for p in model.riem_enc.parameters(): p.requires_grad=True

    # Evaluate
    model.eval(); preds,trues=[],[]
    with torch.no_grad():
        for xr,xe,yb in te_dl:
            out = model(xr.to(DEVICE), xe.to(DEVICE), mode='classify')
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
    torch.save(model.state_dict(), ds_ckpt/f'fold_{fold_idx:02d}.pt')
    return np.array(trues), np.array(preds)


def run_loso(subjects_data, n_cls, cfg, ds_name):
    N = len(subjects_data)
    ds_ckpt = CKPT_DIR/ds_name; ds_ckpt.mkdir(exist_ok=True)
    accs,f1s=[],[]; cm_agg=np.zeros((n_cls,n_cls),dtype=int)
    for i in range(N):
        train_subs = [s for j,s in enumerate(subjects_data) if j!=i]
        test_sub   = subjects_data[i]
        print(f'  Fold {i+1:2d}/{N}  test={test_sub["subject"]} ...', end=' ', flush=True)
        t0=time.time()
        res_pkl = ds_ckpt/f'fold_{i:02d}_results.pkl'
        if res_pkl.exists():
            with open(res_pkl,'rb') as f: y_true,y_pred=pickle.load(f)
            print('(cached)', end=' ')
        else:
            y_true,y_pred = train_one_fold(train_subs, test_sub, n_cls, cfg, i, ds_ckpt)
            with open(res_pkl,'wb') as f: pickle.dump((y_true,y_pred),f)
        acc=accuracy_score(y_true,y_pred)
        f1 =f1_score(y_true,y_pred,average='macro',zero_division=0)
        cm_agg += confusion_matrix(y_true,y_pred,labels=range(n_cls))
        accs.append(acc); f1s.append(f1)
        print(f'acc={acc:.3f}  f1={f1:.3f}  ({time.time()-t0:.0f}s)')
    return dict(acc_mean=float(np.mean(accs)), acc_std=float(np.std(accs)),
                f1_mean=float(np.mean(f1s)),   f1_std=float(np.std(f1s)),
                per_fold_acc=accs, cm=cm_agg)


def run_baselines(subjects_data, n_cls):
    N=len(subjects_data); results={}
    for name,is_cov in [('CSP+LDA',False),('MDM',True),('SVM_Riem',False)]:
        accs=[]
        for i in range(N):
            tr=[s for j,s in enumerate(subjects_data) if j!=i]; te=subjects_data[i]
            covs_tr=np.concatenate([s['covs'] for s in tr]); covs_te=te['covs']
            y_tr=np.concatenate([s['y'] for s in tr]); y_te=te['y']
            try:
                if name=='MDM':
                    clf=MDM(metric='riemann'); clf.fit(covs_tr,y_tr); p=clf.predict(covs_te)
                elif name=='SVM_Riem':
                    ts=TangentSpace(metric='riemann').fit(covs_tr)
                    clf=make_pipeline(StandardScaler(),SVC(kernel='rbf',C=1.0))
                    clf.fit(ts.transform(covs_tr),y_tr); p=clf.predict(ts.transform(covs_te))
                else:
                    flat_tr=covs_tr.reshape(len(covs_tr),-1); flat_te=covs_te.reshape(len(covs_te),-1)
                    clf=make_pipeline(StandardScaler(),LinearDiscriminantAnalysis())
                    clf.fit(flat_tr,y_tr); p=clf.predict(flat_te)
                accs.append(accuracy_score(y_te,p))
            except: accs.append(1.0/n_cls)
        results[name]=dict(acc_mean=float(np.mean(accs)), acc_std=float(np.std(accs)))
        print(f'    {name:<12}: {np.mean(accs)*100:.1f} +/- {np.std(accs)*100:.1f}%')
    return results

print('Training functions defined.')

Training functions defined.


In [8]:
# ── Cell 8: Data loading via MOABB ────────────────────────────

def load_moabb_dataset(ds_name, cfg):
    from moabb.datasets import PhysionetMI, BNCI2014_001, Lee2019_MI
    from moabb.paradigms import MotorImagery
    cls_map = {'PhysionetMI':PhysionetMI, 'BNCI2014_001':BNCI2014_001, 'Lee2019_MI':Lee2019_MI}
    dataset  = cls_map[cfg['moabb_class']]()
    paradigm_kwargs = dict(fmin=cfg['fmin'], fmax=cfg['fmax'],
                           tmin=cfg['tmin'], tmax=cfg['tmax'], resample=cfg['resample'])
    if cfg.get('events'):
        paradigm_kwargs['events']    = cfg['events']
        paradigm_kwargs['n_classes'] = len(cfg['events'])
    else:
        paradigm_kwargs['n_classes'] = cfg['n_classes']
    paradigm = MotorImagery(**paradigm_kwargs)
    subjects = dataset.subject_list[:min(cfg['n_subjects'], len(dataset.subject_list))]
    print(f'  Loading {len(subjects)} subjects for {ds_name} ...')
    X, y_str, meta = paradigm.get_data(dataset, subjects=subjects)
    le = LabelEncoder()
    y_int = le.fit_transform(y_str).astype(np.int64)
    actual_n_cls = int(len(np.unique(y_int)))
    if actual_n_cls != cfg['n_classes']:
        print(f'  [INFO] Actual classes: {actual_n_cls} (config: {cfg["n_classes"]})')
        cfg = {**cfg, 'n_classes': actual_n_cls}
    all_data=[]
    for sub in subjects:
        mask = meta['subject'].astype(str)==str(sub)
        if not mask.any(): continue
        X_sub = X[mask].astype(np.float32)
        y_sub = y_int[mask]
        mu=X_sub.mean(2,keepdims=True); std=X_sub.std(2,keepdims=True)+1e-8
        X_sub=(X_sub-mu)/std
        covs=Covariances(estimator='oas').fit_transform(X_sub)
        all_data.append(dict(X_raw=X_sub, covs=covs, y=y_sub,
                              n_channels=X_sub.shape[1], n_times=X_sub.shape[2],
                              subject=str(sub)))
        print(f'    sub-{sub}: {X_sub.shape[0]} epochs  {X_sub.shape[1]}ch x {X_sub.shape[2]}t  classes={np.bincount(y_sub, minlength=cfg["n_classes"])}')
    print(f'  Loaded {len(all_data)} subjects  ({cfg["n_classes"]} classes)')
    return all_data, cfg

print('Data loader defined.')

Data loader defined.


In [ ]:
# ── Cell 9: Run all datasets ──────────────────────────────────
# Thinking Out Loud results from verification_report.txt (already tested)
all_results = {
    'ThinkingOutLoud': dict(
        acc_mean=0.5833, acc_std=0.0458, f1_mean=0.5722, f1_std=0.0534,
        per_fold_acc=[], n_subjects=10, n_classes=4, chance=0.25),
}
baseline_results = {
    'ThinkingOutLoud': {
        'CSP+LDA':  {'acc_mean':0.250,'acc_std':0.000},
        'MDM':      {'acc_mean':0.257,'acc_std':0.022},
        'SVM_Riem': {'acc_mean':0.242,'acc_std':0.033},
    },
}

for ds_name, ds_cfg in DATASET_REGISTRY.items():
    print(f'\n{"-"*60}')
    print(f'DATASET: {ds_name}  ({ds_cfg["n_subjects"]} subjects, {ds_cfg["n_classes"]} classes)')
    print(f'{"-"*60}')

    cache_pkl = RESULTS_DIR / f'{ds_name}_summary.pkl'
    if cache_pkl.exists():
        with open(cache_pkl,'rb') as f: cached=pickle.load(f)
        all_results[ds_name]      = cached['model']
        baseline_results[ds_name] = cached.get('baselines',{})
        print(f'  [CACHE] acc={cached["model"]["acc_mean"]*100:.1f}%')
        continue

    subjects_data, ds_cfg = load_moabb_dataset(ds_name, ds_cfg)
    if len(subjects_data) < 3:
        print(f'  Skipping: only {len(subjects_data)} subjects'); continue

    n_cls = ds_cfg['n_classes']
    print('  Running baselines ...')
    bl = run_baselines(subjects_data, n_cls)
    baseline_results[ds_name] = bl

    print('  Running EEG-ContrastNet LOSO ...')
    loso = run_loso(subjects_data, n_cls, CFG, ds_name)

    res = dict(**loso, n_subjects=len(subjects_data), n_classes=n_cls, chance=1.0/n_cls)
    all_results[ds_name] = res

    with open(cache_pkl,'wb') as f: pickle.dump({'model':res,'baselines':bl},f)
    print(f'  DONE: acc={res["acc_mean"]*100:.1f} +/- {res["acc_std"]*100:.1f}%  F1={res["f1_mean"]:.3f}')

print('\n=== ALL DATASETS COMPLETE ===')


------------------------------------------------------------
DATASET: PhysionetMI  (15 subjects, 4 classes)
------------------------------------------------------------
  Loading 15 subjects for PhysionetMI ...
Attempting to create new mne-python configuration file:
/root/.mne/mne-python.json
Could not read the /root/.mne/mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)


  0%|                                              | 0.00/2.60M [00:00<?, ?B/s]

SHA256 hash of downloaded file: 3d161f88e1c00632585287d2ce584c2bc0f08862438eb255ea8723e00fac693d
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


Used Annotations descriptions: [np.str_('left_hand'), np.str_('rest'), np.str_('right_hand')]


  0%|                                              | 0.00/2.60M [00:00<?, ?B/s]

...

SHA256 hash of downloaded file: 67233630176aa19faa4133355be20476cc8f571d9327943b9d35a73ecb96f519
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


Used Annotations descriptions: [np.str_('feet'), np.str_('hands'), np.str_('rest')]


  0%|                                              | 0.00/2.56M [00:00<?, ?B/s]

SHA256 hash of downloaded file: 3f57b464e13807f30aacbfc2f6780bc02bf8faecea883725683177416503c875
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


Used Annotations descriptions: [np.str_('feet'), np.str_('hands'), np.str_('rest')]
    sub-1: 90 epochs  64ch x 642t  classes=[24 21 23 22]
    sub-2: 90 epochs  64ch x 642t  classes=[21 24 23 22]
    sub-3: 90 epochs  64ch x 642t  classes=[24 21 23 22]
    sub-4: 90 epochs  64ch x 642t  classes=[23 22 23 22]
    sub-5: 90 epochs  64ch x 642t  classes=[22 23 21 24]
    sub-6: 90 epochs  64ch x 642t  classes=[23 22 24 21]
    sub-7: 90 epochs  64ch x 642t  classes=[23 22 23 22]
    sub-8: 90 epochs  64ch x 642t  classes=[22 23 22 23]
    sub-9: 90 epochs  64ch x 642t  classes=[22 23 24 21]
    sub-10: 90 epochs  64ch x 642t  classes=[22 23 24 21]
    sub-11: 90 epochs  64ch x 642t  classes=[23 22 23 22]
    sub-12: 90 epochs  64ch x 642t  classes=[23 22 21 24]
    sub-13: 90 epochs  64ch x 642t  classes=[23 22 23 22]
    sub-14: 90 epochs  64ch x 642t  classes=[21 24 22 23]
    sub-15: 90 epochs  64ch x 642t  classes=[23 22 23 22]
  Loaded 15 subjects  (4 classes)
  Running baselines .

In [3]:
# ── Cell 10: Comparison table ─────────────────────────────────
print('='*90)
print('EEG-ContrastNet -- Cross-Dataset Comparison (LOSO)')
print('='*90)
print(f'{"Dataset":<18} {"Subs":>5} {"Cls":>4} {"Chance":>7} {"ContrastNet":>13}  {"CSP+LDA":>9}  {"MDM":>9}  {"SVM-Riem":>9}  {"Delta":>8}')
print('-'*90)
for ds_name, r in all_results.items():
    chance = r['chance']; acc = r['acc_mean']; std = r['acc_std']
    delta  = (acc - chance)*100
    bl     = baseline_results.get(ds_name, {})
    def fmt(k):
        return f"{bl[k]['acc_mean']*100:.1f}" if k in bl else '(known)'
    print(f'{ds_name:<18} {r["n_subjects"]:>5} {r["n_classes"]:>4} '
          f'{chance*100:>6.1f}%  '
          f'{acc*100:>7.1f}+/-{std*100:<4.1f}  '
          f'{fmt("CSP+LDA"):>9}  {fmt("MDM"):>9}  {fmt("SVM_Riem"):>9}  '
          f'{delta:>+7.1f}pp')
print('='*90)

EEG-ContrastNet -- Cross-Dataset Comparison (LOSO)
Dataset             Subs  Cls  Chance   ContrastNet    CSP+LDA        MDM   SVM-Riem     Delta
------------------------------------------------------------------------------------------


NameError: name 'all_results' is not defined

In [ ]:
# ── Cell 11: Comparison figures ───────────────────────────────
ds_names = list(all_results.keys())
palette  = ['#1565C0','#2E7D32','#E65100','#6A1B9A'][:len(ds_names)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: grouped bar
ax  = axes[0]
x   = np.arange(len(ds_names))
w   = 0.18
methods = [('EEGContrastNet','EEG-ContrastNet','#1565C0'),
           ('CSP+LDA','CSP+LDA','#78909C'),
           ('MDM','MDM','#546E7A'),
           ('SVM_Riem','SVM-Riem','#37474F')]
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]
for (mkey, mlabel, mcol), off in zip(methods, offsets):
    vals=[]
    for ds in ds_names:
        if mkey=='EEGContrastNet':
            vals.append(all_results[ds]['acc_mean']*100)
        else:
            vals.append(baseline_results.get(ds,{}).get(mkey,{}).get('acc_mean',0)*100)
    ax.bar(x+off, vals, w, color=mcol, alpha=0.85, label=mlabel)
for xi, ds in enumerate(ds_names):
    ch = all_results[ds]['chance']*100
    ax.hlines(ch, xi-2*w, xi+2*w, colors='red', linestyles='--', linewidth=1.5,
              label='Chance' if xi==0 else '')
ax.set_xticks(x); ax.set_xticklabels([d.replace('_',' ') for d in ds_names], fontsize=9)
ax.set_ylabel('Accuracy (%)'); ax.set_title('Classification Accuracy by Method & Dataset', fontweight='bold')
ax.legend(fontsize=8); ax.set_ylim(0,100); ax.grid(axis='y', alpha=0.3)

# Right: delta above chance
ax2    = axes[1]
deltas = [(all_results[d]['acc_mean']-all_results[d]['chance'])*100 for d in ds_names]
errs   = [all_results[d]['acc_std']*100 for d in ds_names]
bars   = ax2.bar(x, deltas, 0.5, yerr=errs, capsize=5, color=palette, alpha=0.85)
for bar, delta in zip(bars, deltas):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{delta:+.1f}pp', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Chance level')
ax2.set_xticks(x); ax2.set_xticklabels([d.replace('_',' ') for d in ds_names], fontsize=9)
ax2.set_ylabel('Improvement above chance (pp)')
ax2.set_title('EEG-ContrastNet Delta above Chance', fontweight='bold')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
out_fig = RESULTS_DIR/'comparison_accuracy.png'
plt.savefig(out_fig, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', out_fig)

In [ ]:
# ── Cell 12: Confusion matrices ───────────────────────────────
trained = {d:r for d,r in all_results.items() if 'cm' in r}
if trained:
    fig, axes = plt.subplots(1, len(trained), figsize=(5*len(trained), 4))
    if len(trained)==1: axes=[axes]
    for ax, (ds_name, r) in zip(axes, trained.items()):
        cm   = r['cm'].astype(float)
        cm_n = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
        names = DATASET_REGISTRY.get(ds_name, {}).get('class_names', [str(i) for i in range(r['n_classes'])])
        sns.heatmap(cm_n, annot=True, fmt='.2f', ax=ax, cmap='Blues',
                    xticklabels=names, yticklabels=names, vmin=0, vmax=1, cbar=False)
        ax.set_title(f'{ds_name}\nacc={r["acc_mean"]*100:.1f}%', fontsize=10)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    out_cm = RESULTS_DIR/'confusion_matrices.png'
    plt.savefig(out_cm, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved:', out_cm)

In [ ]:
# ── Cell 13: Save text report ─────────────────────────────────
lines = ['='*70, 'MULTI-DATASET EEG-ContrastNet REPORT', '='*70, '']
for ds_name, r in all_results.items():
    chance = r['chance']
    lines += [
        f'Dataset     : {ds_name}',
        f'  Subjects  : {r["n_subjects"]}',
        f'  Classes   : {r["n_classes"]}  (chance = {chance*100:.1f}%)',
        f'  EEG-ContrastNet : {r["acc_mean"]*100:.1f} +/- {r["acc_std"]*100:.1f}%  F1={r.get("f1_mean",0):.3f}',
        f'  Delta above chance: +{(r["acc_mean"]-chance)*100:.1f}pp',
    ]
    if ds_name in baseline_results:
        for bname, br in baseline_results[ds_name].items():
            lines.append(f'  {bname:<12}: {br["acc_mean"]*100:.1f} +/- {br["acc_std"]*100:.1f}%')
    if 'per_fold_acc' in r and r['per_fold_acc']:
        lines.append('  Per-fold: ' + '  '.join(f'{a*100:.1f}' for a in r['per_fold_acc']))
    lines.append('')
lines.append('='*70)
report_path = RESULTS_DIR/'multi_dataset_report.txt'
report_path.write_text('\n'.join(lines))
print('Report saved to:', report_path)
print('\n'.join(lines))